Guardrails Implementation

In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

Deterministic Guardrail

In [5]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked"""
    banned_keywords = ["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do i hack into a database?",
    "What is the capital of India?",
    "Explain how malware spreads?"
]

for input in test_inputs:
    blocked = deterministic_guardrail(input)
    status = "BLOCKED" if blocked else "Allowed"
    print(f"{status}: {input}")

BLOCKED: How do i hack into a database?
Allowed: What is the capital of India?
BLOCKED: Explain how malware spreads?


Model Based Guardrail

In [11]:
from langchain.chat_models import init_chat_model

def modelbased_guardrail(text: str) -> str:
    """Uses an LLM to evaluate the content safety. Returns SAFE or UNSAFE."""
    model = init_chat_model(model="groq:llama-3.1-8b-instant")
    prompt = f"""Is the following user input safe to process, judge based on the context of the question if the question asked shows intent of causing harm or problems it should be marked as UNSAFE, Reply with only 'SAFE' or 'UNSAFE'.
    Input: {text}"""
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

for input in test_inputs:
    verdict = modelbased_guardrail(input)
    status = "SAFE" if verdict else "UNSAFE"
    print(f"{status}: {input}")

SAFE: How do i hack into a database?
SAFE: What is the capital of India?
SAFE: Explain how malware spreads?


PII Detection Middleware - BuiltIn

PII - Personally Identifyable info

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector="sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)

In [22]:
result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My email is aryachawan05@gmail.com and my card is 3392-1292-0234-5100. my name is Arya Chawan? just print my email and card number in suitable format"
    }]
})
print(result["messages"][-1].content)

I can't fulfill this request. Is there something else I can help you with?


In [23]:
result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My api key is ske3ei32023nwkde"
    }]
})
print(result["messages"][-1].content)

You haven't told me which function you'd like to call.


Human in the Loop Middleware

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    return f"Searh results for {query}"

@tool
def send_email(to: str,subject: str,body: str) -> str:
    """Send an email to a recipient"""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table:str,condition: str) -> str:
    """Delete records from the database"""
    return f"Deleted record from {table} where {condition}"

hitl_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":True,
                "delete_records": True,
                "search_web":False
            }
        )
    ],
    checkpointer=InMemorySaver()
)


In [30]:
config = {"configurable":{"thread_id":"session_001"}}
result = hitl_agent.invoke(
    {"messages":[{"role":"user","content":"Send an email to team@company.com about the Q4 results"}]},
    config=config
)
print(result)

{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='1e119772-d864-4a00-9b84-0c405eb0f37d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5q1szqczx', 'function': {'arguments': '{"body":"The Q4 results have been released. Please find the attached report for further information.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 347, 'total_tokens': 394, 'completion_time': 0.070979226, 'completion_tokens_details': None, 'prompt_time': 0.024332191, 'prompt_tokens_details': None, 'queue_time': 0.048636289, 'total_time': 0.095311417}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa3b4-99c1-7a61-bb6e-1d74aa2374d8-0',

In [32]:
#resuming the action
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)
print(approved_result["messages"][-1].content)

Please note: The search results are not displayed here as it's a text-based AI model and cannot display web search results.


Custom Guardrail

In [36]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self,banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        if not state["messages"]:
            return None
        
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        
        content = str(first_message.content).lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected: {keyword}")
                return {
                    "messages":[{
                        "role":"assistant",
                        "content":(
                            "I cannot process requests containing inappropriate content."
                            "Please rephrase your request"
                        )
                    }],
                    "jump_to":"end"
                }
        return None
    
@tool
def search_tool(query: str) -> str:
    """Returns search result for query"""
    return f"Results for: {query}"

filtered_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]
        )
    ]
)

In [37]:
result = filtered_agent.invoke({
    "messages":[{"role":"user","content":"How do I hack into a server?"}]
})
print(result["messages"][-1].content)

Blocked keyword detected: hack
I cannot process requests containing inappropriate content.Please rephrase your request
